# HW100 · 2/3 — Run

Three modes, same output format (`results/hw100/<mode>/<method>.npz`, an int8 array `bits` of shape (shots, 120)):

* `model`  — samples from the depolarizing model (instant, to test notebook 3).
* `aer`    — gate-level noisy simulation (FakeFez noise model), each 11-qubit block simulated separately; takes minutes.
* `ibm`    — **real hardware**. Needs your key file. One job with all seven 120-qubit circuits.

Run notebook 1 first (ideally with `USE_LIVE_BACKEND = True` right before this one).

In [ ]:
import os, sys, json, pickle, time, warnings; warnings.filterwarnings('ignore')
root = os.path.abspath('.')
while not os.path.isdir(os.path.join(root, 'negsearch')): root = os.path.dirname(root)      # repository root
os.chdir(root); sys.path.insert(0, root)
import numpy as np
from negsearch import hw100 as H
OUT = 'hardware_circuits/hw100'; RES = 'results/hw100'; os.makedirs(OUT, exist_ok=True); os.makedirs(RES, exist_ok=True)
print('repository root:', root)

In [ ]:
# ---------------- configuration ----------------
MODE = 'model'                    # 'model' | 'aer' | 'ibm'
SHOTS = 4000
APIKEY = 'apikey_personal.json'   # local file {"apikey": "..."}; never committed, never in the zip
BACKEND_NAME = 'ibm_fez'          # any Heron device with >= 130 usable qubits; must be the one used in notebook 1
INSTANCE = None
DD = True                         # dynamical decoupling
TWIRL = True                      # Pauli twirling of the CZ gates (32 randomizations)
RETRIEVE_JOB_ID = None            # set to a job id string to fetch the results of a job submitted earlier
AER_SHOTS, AER_BLOCKS = 500, None # aer mode: shots per block, and list of block ids (None = all 12)

In [ ]:
from qiskit import qpy
meta = json.load(open(f'{OUT}/meta.json')); D = pickle.load(open(f'{OUT}/instance.pkl', 'rb')); inst, ideal = D['inst'], D['ideal']
methods = meta['methods']
big = {m: qpy.load(open(f'{OUT}/{m}_120q.qpy', 'rb'))[0] for m in methods}
subs = {m: qpy.load(open(f'{OUT}/{m}_blocks.qpy', 'rb')) for m in methods}
print('circuits loaded:', {m: (c.num_qubits, c.num_clbits) for m, c in big.items()})
dest = f'{RES}/{MODE}'; os.makedirs(dest, exist_ok=True)

In [ ]:
def save(m, bits): np.savez_compressed(f'{dest}/{m}.npz', bits=bits.astype(np.int8))

if MODE == 'model':
    for i, m in enumerate(methods):
        esp = [s['esp'] for s in meta['compile'][m]]
        save(m, H.sample_model(ideal, esp, m, SHOTS, seed=i)); print(m, 'sampled')

elif MODE == 'aer':
    from qiskit_ibm_runtime.fake_provider import FakeFez
    bk = FakeFez()
    for i, m in enumerate(methods):
        print(m)
        save(m, H.aer_blocks(inst, subs[m], meta['regions'][m], bk, shots=AER_SHOTS, blocks=AER_BLOCKS, seed=i))

elif MODE == 'ibm':
    from qiskit_ibm_runtime import SamplerV2 as Sampler
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    svc, backend = H.connect(APIKEY, BACKEND_NAME, INSTANCE)
    print('backend', backend.name, '| status:', backend.status().status_msg, '| queue:', backend.status().pending_jobs)
    if RETRIEVE_JOB_ID:
        job = svc.job(RETRIEVE_JOB_ID)
    else:
        pm = generate_preset_pass_manager(optimization_level=0, backend=backend)       # keeps our layout; only checks ISA / adds nothing
        isa = {m: pm.run(big[m]) for m in methods}
        for m in methods:
            assert H.check_isa(isa[m], backend), f'{m}: not ISA on the live backend (recompile in notebook 1 with USE_LIVE_BACKEND=True)'
        sampler = Sampler(mode=backend)
        if DD: sampler.options.dynamical_decoupling.enable = True; sampler.options.dynamical_decoupling.sequence_type = 'XpXm'
        if TWIRL:
            sampler.options.twirling.enable_gates = True; sampler.options.twirling.num_randomizations = 32
            sampler.options.twirling.shots_per_randomization = max(1, SHOTS // 32)
        job = sampler.run([(isa[m], None, SHOTS) for m in methods])
        json.dump(dict(job_id=job.job_id(), backend=backend.name, shots=SHOTS, dd=DD, twirl=TWIRL, time=time.ctime()), open(f'{dest}/job.json', 'w'))
        print('submitted job', job.job_id(), '(saved to', f'{dest}/job.json)')
    res = job.result()
    for m, r in zip(methods, res):
        counts = r.data[list(r.data.keys())[0]].get_counts()
        save(m, H.counts_to_bits(counts, 120)); print(m, sum(counts.values()), 'shots')
    print('saved. Next: notebook 3 with SOURCE = "ibm".')

### Notes for the real run
* One job, six circuits × `SHOTS`. Estimated QPU time is a few seconds per circuit (the deepest block is about 1000–1400 layers, roughly 0.2–0.4 ms per shot including reset), so the whole job is on the order of a minute or two; the queue dominates.
* If `check_isa` fails, the backend's couplers changed since notebook 1: rerun notebook 1 with `USE_LIVE_BACKEND = True`.
* Twirling on a post-selected experiment is fine (it only randomises the coherent part of the error). Switch `TWIRL`/`DD` off to see their effect; running both settings is a worthwhile control.
* If the job is long in the queue, you can stop the notebook and come back with `RETRIEVE_JOB_ID`.